In [7]:
import pandas as pd
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")
train_data.head()
test_data.head()


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [8]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)
print("% of women survived:", rate_women)
print(sum(women), len(women))


% of women survived: 0.7420382165605095
233 314


In [9]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)
print("% of men survived:", rate_men)

% of men survived: 0.18890814558058924


In [10]:
# 1. 创建一个新特征，标记年龄是否缺失
train_data['Age_Unknown'] = train_data['Age'].isna()
test_data['Age_Unknown'] = test_data['Age'].isna()

# 2. 将年龄分成有意义的组（儿童、成人、老人等）
def create_age_group(age):
    if pd.isna(age):
        return 'Unknown' # 缺失单独成组
    elif age < 12:
        return 'Child'
    elif age < 18:
        return 'Teenager'
    elif age < 65:
        return 'Adult'
    else:
        return 'Elderly'

train_data["Age_Group"] = train_data['Age'].apply(create_age_group)
test_data['Age_Group'] = test_data['Age'].apply(create_age_group)

In [11]:
train_data['Age_Unknown'] = train_data['Age'].isna()

test_data['Age_Unknown'] = test_data['Age'].isna()
#创造一个循环，来将不同年龄区间变成字符串
def creat_age_group(age):
    if pd.isna(age):
        return 'Unknow' 
    elif age < 12:
        return 'Child'
    elif age < 18:
        return 'Teenager'
    elif age < 65:
        return 'Adult'
    else:
        return 'Elderly'

train_data["Age_Group"] = train_data['Age'].apply(creat_age_group)
test_data["Age_Group"] = test_data['Age'].apply(creat_age_group)


In [12]:
from sklearn.ensemble import RandomForestClassifier
# 1. 定义特征列表
features = ["Pclass", "Sex", "SibSp", "Parch", "Fare", "Embarked", "Age_Group", "Age_Unknown"]

# 2. 【关键】统一过滤：同时删除训练数据中，在features列表上有任何缺失的行
# 这行代码会同时清洗 train_data 和对应的 y，确保它们行数一致
clean_train_data = train_data.dropna(subset=features)
y = clean_train_data["Survived"]  # 从清洗后的数据中提取标签

# 3. 用清洗后的数据创建特征矩阵
X = pd.get_dummies(clean_train_data[features])
X_test = pd.get_dummies(test_data[features])  # 测试集仍需处理，但不能直接删除行

# 4. 确保测试集无NaN（用0填充）
X_test = X_test.fillna(0)

# 5. 对齐列（防止因测试集缺少某个类别导致列数不同）
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# 后续建模代码保持不变
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print(f"训练有效样本数: {len(X)}")
print("Your submission was successfully saved!")

训练有效样本数: 889
Your submission was successfully saved!
